In [18]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
import pinecone
from sentence_transformers import SentenceTransformer

In [19]:
files = pd.read_csv('course_descriptions.csv', encoding = 'ANSI')

In [20]:
def create_course_descriptions(row):
    return f''' Course Name: {row['course_name']},
               Course Slug: {row['course_slug']},
               Course Technology: {row['course_technology']},
               Course Topic: {row['course_topic']}
               '''

In [21]:
pd.set_option('display.max_row', 106)
files['course_description_new'] = files.apply(create_course_descriptions, axis=1)
print(files['course_description_new'].head())

0     Course Name: Introduction to Tableau,\n      ...
1     Course Name: The Complete Data Visualization ...
2     Course Name: Introduction to R Programming,\n...
3     Course Name: Data Preprocessing with NumPy,\n...
4     Course Name: Introduction to Data and Data Sc...
Name: course_description_new, dtype: object


In [22]:
load_dotenv(find_dotenv(), override=True)

True

In [23]:
pc =Pinecone(api_key=os.environ['PINECONE_API_KEY'])

In [24]:
index_name = 'my-index'
dimension = 384
metric = 'cosine'

In [25]:
if index_name not in [index_name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f'Index {index_name} deleted.')
else:
    print(f'{index_name} does not exist.')

my-index does not exist.


In [26]:
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric=metric,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)


In [27]:

index = pc.Index(index_name)
print(f'Index {index_name} is ready to use.')

Index my-index is ready to use.


In [28]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [29]:
def create_embeddings(row):
    combined_text = ' '.join([str(row[field]) for field in ['course_description', 'course_description_new', 'course_description_short']])
    embedding = model.encode(combined_text, show_progress_bar=False)
    return embedding

In [30]:
files['embedding'] = files.apply(create_embeddings, axis=1)

In [31]:
pc.list_indexes()

[
    {
        "name": "text",
        "metric": "cosine",
        "host": "text-nq587m6.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "my-index-2",
        "metric": "euclidean",
        "host": "my-index-2-nq587m6.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
            

In [32]:
vectors_to_upsert = [(str(row['course_name']), row['embedding'].tolist()) for _, row in files.iterrows()]
index.upsert(vectors=vectors_to_upsert)

print('Data upserted to Pinecone index successfully.')

Data upserted to Pinecone index successfully.


In [33]:
query = "clustering"
query_embedding = model.encode(query, show_progress_bar=False).tolist()

In [34]:
query_results = index.query(
    vector= (query_embedding),
    top_k=12,
    include_values=True
)

In [35]:
query_results

QueryResponse(matches=[{'id': 'Machine Learning in Excel',
 'score': 0.363689482,
 'values': [-0.0191816371,
            -0.0306801163,
            -0.0228996873,
            -0.0267805643,
            -0.0261368565,
            -0.0177778509,
            -0.0482332334,
            -0.0631067231,
            0.00167293684,
            0.0203194935,
            -0.030152617,
            -0.0342542939,
            0.0674666241,
            -0.0367042981,
            0.00723213144,
            0.0324933492,
            -0.00416697608,
            -0.00101283449,
            -0.00432655262,
            -0.0611976907,
            0.0797586292,
            0.0316995531,
            -0.0494814515,
            -0.031139642,
            0.0326453559,
            0.0364636369,
            0.0620523281,
            0.00785914343,
            -0.0439172834,
            -0.0333911553,
            -0.0434330069,
            0.0113393329,
            0.0139435381,
            0.0512274951,
          

In [36]:
for match in query_results['matches']:
    print(f"Course Name: {match['id']}, Score: {match['score']}")

Course Name: Machine Learning in Excel, Score: 0.363689482
Course Name: Machine Learning with K-Nearest Neighbors, Score: 0.337460071
Course Name: Machine Learning in Python, Score: 0.292277366
Course Name: Customer Churn Analysis with SQL and Tableau, Score: 0.271809608
Course Name: Linear Algebra and Feature Selection, Score: 0.265286475
Course Name: Growth Analysis with SQL, Python, and Tableau  , Score: 0.254835159
Course Name: Machine Learning with Naive Bayes, Score: 0.237872615
Course Name: Machine Learning with Support Vector Machines, Score: 0.235735908
Course Name: Fashion Analytics with Tableau, Score: 0.233521
Course Name: Customer Engagement Analysis with SQL and Tableau, Score: 0.217132613
Course Name: Introduction to Industry and Company Analysis, Score: 0.201859489
Course Name: Data Preprocessing with NumPy, Score: 0.19825843


In [37]:
df = pd.read_csv('course_section_descriptions.csv', encoding = 'ANSI')

In [39]:
df.head()

,course_id,course_name,course_slug,course_description,course_description_short,course_technology,course_topic,course_instructor_quote,section_id,section_name,section_description
0,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,9,Introduction to Tableau,While Tableau is an indispensable tool in the ...
1,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,10,Tableau Functionalities,"In this section, you will create your first Ta..."
2,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,11,The Tableau Exercise,This section is a practical example that will ...
3,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,12,Introduction,"In this section, you will learn about the impo..."
4,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,13,Setting Up the Environments,"Here, we set up different environments for the..."


In [40]:
df['unique_id'] = df['course_id'].astype(str) + '_' + df['section_id'].astype(str)

In [41]:
df['metadata'] = df.apply(lambda row: {
    'course_name': row['course_name'],
    'section_name': row['section_name'],
    'section_description': row['section_description'],
}, axis=1)

In [47]:
def create_new_embeddings(row):
    combined_text = f''' {row['course_name']} {row['course_technology']}
    {row['course_description']}{row['section_name']}{row['section_description']} '''
    return model1.encode(combined_text, show_progress_bar=False)
    

In [48]:
model1 = SentenceTransformer('all-MiniLM-L6-v2')

In [49]:
df['embedding'] = df.apply(create_new_embeddings, axis=1)

In [54]:
df.columns.tolist()

['course_id',
 'course_name',
 'course_slug',
 'course_description',
 'course_description_short',
 'course_technology',
 'course_topic',
 'course_instructor_quote',
 'section_id',
 'section_name',
 'section_description',
 'unique_id',
 'metadata',
 'embedding']

In [53]:
vector_to_upsert = [(row['unique_id'], row['embedding'].tolist(), row['metadata']) for index, row in df.iterrows()]

In [55]:
index.upsert(vectors=vector_to_upsert)
print("Data upserted to Pinecone index successfully.")

Data upserted to Pinecone index successfully.
